In [64]:
import gymnasium as gym
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import random

from google.colab import drive
drive.mount('/content/drive')

sys.path.append('/content/drive/MyDrive/wordle-solver-master/')
import deep_rl.wordle.wordle

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [69]:
class NN(nn.Module):
    def __init__(self,input_dim,output_dim):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim,16),
            nn.Sigmoid(),
            nn.Linear(16,32),
            nn.Sigmoid(),
            nn.Linear(32,output_dim),
            nn.Softmax(dim=1)
        )
    def forward(self,x):
        return self.model(x)

class BaselineNN(nn.Module):
    def __init__(self,input_dim):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim,16),
            nn.Sigmoid(),
            nn.Linear(16,32),
            nn.Sigmoid(),
            nn.Linear(32,1)
        )
    def forward(self,x):
        return self.model(x)

In [71]:
learning_rate = 1e-2
gamma = 0.99

seed = 1

torch.manual_seed(seed)
random.seed(seed)
np.random.seed(seed)

env_name = "WordleEnv10-v0"
env = gym.make(env_name)

input_dim = env.observation_space.shape[0]
output_dim = env.action_space.n

model = NN(input_dim, output_dim)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

baseline_model = BaselineNN(input_dim)
baseline_optimizer = torch.optim.Adam(baseline_model.parameters(), lr=learning_rate)

episodes = 10000

all_rewards = []

record = 100

for episode in range(episodes):
    state,_ = env.reset()
    state = torch.FloatTensor(state).unsqueeze(0)

    log_ps = []
    rewards = []
    values = []

    done = False
    while not done:
        action_p = model(state)
        dist = torch.distributions.Categorical(action_p)
        action = dist.sample()

        value = baseline_model(state)
        values = values + [value]

        next_state, reward, terminated, truncated, _ = env.step(action.item())
        done = terminated or truncated

        log_p = dist.log_prob(action)


        log_ps.append(log_p)
        rewards = rewards + [reward]

        state = torch.FloatTensor(next_state).unsqueeze(0)

    losses = []
    baseline_losses = []
    T = len(rewards)

    if T == 1:
      continue

    for t in range(T-1):
        G = 0.0
        for k in range(t+1,T):
            G += (gamma**(k-t-1)) * rewards[k]
        loss = -log_ps[t] * (G - values[t].detach())
        losses = losses + [loss]
        baseline_loss = (G - values[t])**2
        baseline_losses = baseline_losses + [baseline_loss]

    final_loss = torch.stack(losses).mean()
    baseline_loss = torch.stack(baseline_losses).mean()

    optimizer.zero_grad()
    baseline_optimizer.zero_grad()
    final_loss.backward()
    baseline_loss.backward()
    optimizer.step()
    baseline_optimizer.step()

    all_rewards = all_rewards + [sum(rewards)]

    if episode % record == 0 and episode>0:
        mean_reward = sum(all_rewards[-record:]) / record
        print(mean_reward)



env.close()

-3.8
-6.8
-9.2
-9.4
-10.0
-9.8
-9.8
-10.0
-9.6
-9.6
-10.0
-10.0
-9.8
-10.0
-10.0
-10.0
-10.0
-10.0
-10.0
-9.8
-7.8
-5.6
-7.2
-5.0
-6.2
-8.0
-8.6
-7.2
-7.4
-6.8
-7.0
-7.8
-7.6
-8.2
-6.8
-5.6
-5.6
-3.0
-4.2
-2.4
-2.2
-4.8
-4.2
-3.6
-3.0
-1.4
-2.6
-2.4


KeyboardInterrupt: 